# Notebook 03 — Statistical Analysis
**Thesis:** Chapter 7 | **Input:** `data/2_events.csv` | **Output:** `data/3_*.csv`

---

## Maps to thesis tables

| Table | Content | Thesis section |
|-------|---------|---------------|
| 7.1 | Per-device campaign statistics | §7.1.1 |
| 7.2 | Burst loss distribution (E16) | §7.2.1 |
| 7.3 | Weekday vs weekend per device (E2) | §7.3.1 |
| 7.4 | Season + SF tier PDR with 95% CI | §7.3.3, §7.6 |
| 7.5 | CO₂ tier PDR + burst rate + sample% | §7.4 |
| 7.6 | SF PDR + burst rate + p vs SF7 | §7.6.1 |
| 7.7 | Joint SF tier × CO₂ tier | §7.6.2 |
| 7.8 | Logistic regression OR (loss + burst) | §7.7 |

## Dataset
All analysis uses `df_link` — the radio-layer filtered dataset:
```python
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']]
```
This ensures PDR variation reflects radio-channel conditions only.

## 0 · Imports & Load

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data')
df = pd.read_csv(DATA_DIR / '2_events.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id', 'time']).reset_index(drop=True)

# radio-layer filtered dataset — excludes infrastructure outages and SF artifacts
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']].copy().reset_index(drop=True)

print(f'Full dataset  : {len(df):,} rows')
print(f'df_link       : {len(df_link):,} rows  (filtered for event analysis)')
print(f'Excluded      : {len(df)-len(df_link):,} rows  (outages + SF artifacts)')
print(f'Devices       : {sorted(df_link["device_id"].unique())}')

Full dataset  : 1,217,313 rows
df_link       : 1,156,407 rows  (filtered for event analysis)
Excluded      : 60,906 rows  (outages + SF artifacts)
Devices       : ['ED0', 'ED1', 'ED2', 'ED3', 'ED4', 'ED5']


## 1 · Statistical Helper Functions  *(§5.5)*

Three helpers used throughout:
- `pdr()` — PDR from a group of rows
- `burst_rate()` — fraction of intervals with loss ≥ 3 (burst)
- `mannwhitney()` — two-sided Mann-Whitney U + rank-biserial r
- `bootstrap_ci()` — 95% CI on PDR via bootstrap resampling

In [2]:
def pdr(grp: pd.DataFrame) -> float:
    """PDR = received / (received + reconstructed radio losses)."""
    rx   = len(grp)
    lost = int(grp['mac_to_radio_loss'].sum())
    return rx / (rx + lost) * 100 if (rx + lost) > 0 else np.nan

def burst_rate(grp: pd.DataFrame) -> float:
    """Fraction of intervals where mac_to_radio_loss >= 3 (burst threshold B=3)."""
    return (grp['mac_to_radio_loss'] >= 3).mean() * 100

def mannwhitney(a: np.ndarray, b: np.ndarray):
    """Two-sided Mann-Whitney U. Returns (U, p_value, rank_biserial_r)."""
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    r    = 1 - (2 * u) / (len(a) * len(b))   # rank-biserial correlation
    return u, p, r

def bootstrap_ci(grp: pd.DataFrame, n_boot: int = 1000, ci: float = 95.0):
    """Bootstrap 95% CI on PDR for a group. Returns (lo, hi)."""
    pdrs = []
    for _ in range(n_boot):
        sample = grp.sample(n=len(grp), replace=True)
        pdrs.append(pdr(sample))
    lo = np.percentile(pdrs, (100 - ci) / 2)
    hi = np.percentile(pdrs, 100 - (100 - ci) / 2)
    return round(lo, 1), round(hi, 1)

TOA_MS = {7: 71.9, 8: 133.6, 9: 246.8, 10: 452.6}

print('Helper functions ready.')
print('pdr() | burst_rate() | mannwhitney() | bootstrap_ci()')

Helper functions ready.
pdr() | burst_rate() | mannwhitney() | bootstrap_ci()


## 2 · Table 7.1 — Campaign-Wide Reconstruction Results  *(§7.1.1)*

Per-device statistics. PDR computed on `df_link` (radio-layer filtered).  
Reset intervals excluded (loss count unknowable at reboot).

**Key observation:** PDR range across devices should be very small (1–3 pp)  
despite vastly different link budgets — the collision-dominated hypothesis.

In [3]:
rows = []
for dev, g in df_link.groupby('device_id'):
    rx    = len(g)
    lost  = int(g['mac_to_radio_loss'].sum())
    total = rx + lost
    p     = rx / total * 100
    resets= int(df[df['device_id']==dev]['is_reset'].sum())
    dist  = int(g['distance'].iloc[0])
    cw    = int(g['c_walls'].iloc[0])
    ww    = int(g['w_walls'].iloc[0])
    rows.append({'Device': dev, 'Dist(m)': dist, 'CW': cw, 'WW': ww,
                 'Received': rx, 'Recon_Lost': lost,
                 'Total_TX': total, 'PDR_pct': round(p, 2), 'Resets': resets})

t71 = pd.DataFrame(rows)
print('TABLE 7.1: Campaign-Wide Reconstruction Results')
print(t71.to_string(index=False))

# campaign total
total_rx   = t71['Received'].sum()
total_lost = t71['Recon_Lost'].sum()
total_tx   = t71['Total_TX'].sum()
camp_pdr   = total_rx / total_tx * 100
pdr_range  = t71['PDR_pct'].max() - t71['PDR_pct'].min()
print(f"\nCampaign total : {total_rx:,} received | {total_lost:,} lost | PDR = {camp_pdr:.2f}%")
print(f"PDR range across devices: {pdr_range:.2f} pp  → evidence for collision-dominated losses")

t71.to_csv(DATA_DIR / '3_table_7_1.csv', index=False)

TABLE 7.1: Campaign-Wide Reconstruction Results
Device  Dist(m)  CW  WW  Received  Recon_Lost  Total_TX  PDR_pct  Resets
   ED0       10   0   0    192030        7185    199215    96.39       4
   ED1        8   1   0    191072        7547    198619    96.20       1
   ED2       23   0   2    193876        6479    200355    96.77       1
   ED3       18   1   2    190340        7652    197992    96.14       3
   ED4       37   0   5    189995        7996    197991    95.96       1
   ED5       40   2   2    199094        6848    205942    96.67       4

Campaign total : 1,156,407 received | 43,707 lost | PDR = 96.36%
PDR range across devices: 0.81 pp  → evidence for collision-dominated losses


## 3 · Table 7.2 — Burst Loss Distribution  *(§7.2.1)*

Distribution of E16 loss types across all devices combined.  
Uses `df_link` — SF artifacts already excluded from `e16_loss_type` classification.

**B=3 threshold:** burst defined as ≥3 consecutive losses (≥3-minute data gap).

In [4]:
# E16 loss type counts on df_link
order = ['no_loss', 'isolated', 'small_burst', 'large_burst']
counts = df_link['e16_loss_type'].value_counts().reindex(order).fillna(0).astype(int)
pcts   = (counts / len(df_link) * 100).round(1)

t72 = pd.DataFrame({'Loss_Type': order, 'Count': counts.values, 'Pct': pcts.values})
print('TABLE 7.2: Burst Loss Distribution (df_link)')
print(t72.to_string(index=False))

burst_intervals = int(counts[['small_burst','large_burst']].sum())
print(f"\nIntervals with any loss     : {int(counts[['isolated','small_burst','large_burst']].sum()):,}  ({pcts[['isolated','small_burst','large_burst']].sum():.1f}%)")
print(f"Intervals with burst (≥3)   : {burst_intervals:,}  ({pcts[['small_burst','large_burst']].sum():.1f}%)")

t72.to_csv(DATA_DIR / '3_table_7_2.csv', index=False)

TABLE 7.2: Burst Loss Distribution (df_link)
  Loss_Type   Count  Pct
    no_loss 1135043 98.2
   isolated   15024  1.3
small_burst    4450  0.4
large_burst    1890  0.2

Intervals with any loss     : 21,364  (1.9%)
Intervals with burst (≥3)   : 6,340  (0.6%)


## 4 · Table 7.3 — Weekday vs Weekend per Device  *(§7.3.1)*

Mann-Whitney U test on `total_tx` distributions within each device group.  
Expected finding: weekday PDR ~7-8 pp lower than weekend across all devices.

In [5]:
rows = []
for dev, g in df_link.groupby('device_id'):
    wk  = g[g['e2_is_weekday'] == 1]
    we  = g[g['e2_is_weekday'] == 0]
    p_wk = pdr(wk)
    p_we = pdr(we)
    _, p_val, r = mannwhitney(wk['total_tx'].values, we['total_tx'].values)
    p_str = '< 0.001' if p_val < 0.001 else f'{p_val:.4f}'
    rows.append({'Device': dev,
                 'Weekday_PDR': round(p_wk, 1),
                 'Weekend_PDR': round(p_we, 1),
                 'Diff_pp': round(p_wk - p_we, 1),
                 'p_value': p_str,
                 'r': round(r, 3)})

# all devices combined
wk_all = df_link[df_link['e2_is_weekday'] == 1]
we_all = df_link[df_link['e2_is_weekday'] == 0]
_, p_all, r_all = mannwhitney(wk_all['total_tx'].values, we_all['total_tx'].values)
rows.append({'Device': 'All', 'Weekday_PDR': round(pdr(wk_all), 1),
             'Weekend_PDR': round(pdr(we_all), 1),
             'Diff_pp': round(pdr(wk_all)-pdr(we_all), 1),
             'p_value': '< 0.001' if p_all < 0.001 else f'{p_all:.4f}',
             'r': round(r_all, 3)})

t73 = pd.DataFrame(rows)
print('TABLE 7.3: Weekday vs Weekend PDR per Device')
print(t73.to_string(index=False))
t73.to_csv(DATA_DIR / '3_table_7_3.csv', index=False)

TABLE 7.3: Weekday vs Weekend PDR per Device
Device  Weekday_PDR  Weekend_PDR  Diff_pp p_value      r
   ED0         96.2         96.9     -0.7 < 0.001 -0.003
   ED1         96.0         96.7     -0.7  0.0396 -0.002
   ED2         96.6         97.1     -0.5  0.9389 -0.000
   ED3         95.9         96.7     -0.7  0.0013 -0.002
   ED4         95.7         96.6     -0.9  0.0242 -0.002
   ED5         96.1         98.1     -2.0 < 0.001 -0.006
   All         96.1         97.0     -0.9 < 0.001 -0.002


## 5 · Table 7.4 — Season + SF Tier PDR with 95% CI  *(§7.3.3, §7.6)*

Campaign-wide PDR by season and by SF tier.  
95% confidence intervals computed by bootstrap (1000 samples).

In [6]:
rows = []

# seasons
for season in ['autumn', 'winter', 'spring']:
    g = df_link[df_link['e6_season'] == season]
    p = round(pdr(g), 1)
    lo, hi = bootstrap_ci(g)
    rows.append({'Condition': f'Season: {season}', 'PDR_pct': p,
                 'CI_lo': lo, 'CI_hi': hi, 'n': len(g)})

# SF tier
for tier in ['low_sf', 'high_sf']:
    g = df_link[df_link['e14_sf_tier'] == tier]
    p = round(pdr(g), 1)
    lo, hi = bootstrap_ci(g)
    label = 'SF tier: Low SF (7-8)' if tier == 'low_sf' else 'SF tier: High SF (9-10)'
    rows.append({'Condition': label, 'PDR_pct': p,
                 'CI_lo': lo, 'CI_hi': hi, 'n': len(g)})

t74 = pd.DataFrame(rows)
print('TABLE 7.4: PDR by Season (E6) and SF Tier (E14) with 95% CI')
print(t74.to_string(index=False))
t74.to_csv(DATA_DIR / '3_table_7_4.csv', index=False)

KeyboardInterrupt: 

## 6 · Table 7.5 — CO₂ Tier PDR + Burst Rate  *(§7.4)*

CO₂ is the primary occupancy proxy. Three-tier comparison with:
- PDR % — packet delivery ratio
- Burst Rate % — fraction of intervals with ≥3 consecutive losses
- Sample % — how much of the data each tier represents

All pairwise comparisons tested with Mann-Whitney U (α = 0.05).

In [7]:
rows = []
for tier in ['background', 'moderate', 'high']:
    g = df_link[df_link['e1_co2_tier'] == tier]
    rows.append({
        'CO2_Tier': tier,
        'PDR_pct':      round(pdr(g), 1),
        'Burst_Rate_pct': round(burst_rate(g), 1),
        'Sample_pct':   round(len(g) / len(df_link) * 100, 1),
        'n':            len(g)
    })

t75 = pd.DataFrame(rows)
print('TABLE 7.5: CO2 Tier PDR and Burst Rate (E1)')
print(t75.to_string(index=False))

# pairwise Mann-Whitney tests
print('\nPairwise Mann-Whitney U tests:')
for t1, t2 in [('background','moderate'), ('background','high'), ('moderate','high')]:
    g1 = df_link[df_link['e1_co2_tier']==t1]
    g2 = df_link[df_link['e1_co2_tier']==t2]
    _, p_val, r = mannwhitney(g1['total_tx'].values, g2['total_tx'].values)
    p_str = '< 0.001' if p_val < 0.001 else f'{p_val:.4f}'
    print(f'  {t1} vs {t2}: p = {p_str}  |  r = {r:.3f}')

t75.to_csv(DATA_DIR / '3_table_7_5.csv', index=False)

TABLE 7.5: CO2 Tier PDR and Burst Rate (E1)
  CO2_Tier  PDR_pct  Burst_Rate_pct  Sample_pct      n
background     96.8             0.3        51.7 597487
  moderate     96.3             0.3        36.0 416678
      high     94.6             0.4        12.3 142242

Pairwise Mann-Whitney U tests:
  background vs moderate: p = < 0.001  |  r = 0.002
  background vs high: p = < 0.001  |  r = 0.010
  moderate vs high: p = < 0.001  |  r = 0.007


## 7 · Table 7.6 — SF PDR + Burst Rate + p vs SF7  *(§7.6.1)*

PDR conditioned on individual spreading factor.  
Burst rate and statistical comparison vs SF7 baseline included.

Expected: monotone decrease in PDR and increase in burst rate with SF.

In [8]:
sf7_tx = df_link[df_link['e13_sf'] == 7]['total_tx'].values

rows = []
for sf in [7, 8, 9, 10]:
    g   = df_link[df_link['e13_sf'] == sf]
    p   = round(pdr(g), 1)
    br  = round(burst_rate(g), 1)
    toa = TOA_MS[sf]
    if sf == 7:
        p_str = '—'
    else:
        _, p_val, _ = mannwhitney(sf7_tx, g['total_tx'].values)
        p_str = '< 0.001' if p_val < 0.001 else f'{p_val:.4f}'
    rows.append({'SF': sf, 'ToA_ms': toa, 'PDR_pct': p,
                 'Burst_Rate_pct': br, 'p_vs_SF7': p_str})

t76 = pd.DataFrame(rows)
print('TABLE 7.6: PDR and Burst Rate by Spreading Factor (E13)')
print(t76.to_string(index=False))
print(f"\nPDR drop SF7 → SF10: {rows[0]['PDR_pct'] - rows[3]['PDR_pct']:.1f} pp")
print(f"Burst rate increase SF7 → SF10: {rows[3]['Burst_Rate_pct'] - rows[0]['Burst_Rate_pct']:.1f} pp")
t76.to_csv(DATA_DIR / '3_table_7_6.csv', index=False)

TABLE 7.6: PDR and Burst Rate by Spreading Factor (E13)
 SF  ToA_ms  PDR_pct  Burst_Rate_pct p_vs_SF7
  7    71.9     98.1             0.1        —
  8   133.6     97.4             0.1  < 0.001
  9   246.8     97.5             0.1  < 0.001
 10   452.6     91.4             1.0  < 0.001

PDR drop SF7 → SF10: 6.7 pp
Burst rate increase SF7 → SF10: 0.9 pp


## 8 · Table 7.7 — Joint SF Tier × CO₂ Tier  *(§7.6.2 — Novel Finding)*

The headline novel finding of this thesis.  
Does the PDR gap between low-SF and high-SF transmissions widen during high-occupancy periods?

**Hypothesis:** Under ALOHA collision dynamics, longer ToA (high SF) suffers  
disproportionately when channel traffic is elevated (high CO₂/occupancy).

In [ ]:
rows = []
for co2_tier in ['background', 'moderate', 'high']:
    for sf_tier in ['low_sf', 'high_sf']:
        g = df_link[(df_link['e14_sf_tier']==sf_tier) &
                    (df_link['e1_co2_tier']==co2_tier)]
        rows.append({'SF_Tier': sf_tier, 'CO2_Tier': co2_tier,
                     'PDR_pct': round(pdr(g), 1), 'n': len(g)})

t77_long = pd.DataFrame(rows)
pivot = t77_long.pivot(index='SF_Tier', columns='CO2_Tier', values='PDR_pct')
pivot = pivot[['background', 'moderate', 'high']]

print('TABLE 7.7: Joint PDR conditioned on SF Tier × CO2 Tier')
print(pivot.to_string())

print('\nSF gap (Low SF - High SF) per CO2 tier:')
for co2_tier in ['background', 'moderate', 'high']:
    gap = pivot.loc['low_sf', co2_tier] - pivot.loc['high_sf', co2_tier]
    print(f'  {co2_tier:<12}: {gap:.1f} pp')

gap_bg = pivot.loc['low_sf','background'] - pivot.loc['high_sf','background']
gap_hi = pivot.loc['low_sf','high']        - pivot.loc['high_sf','high']
print(f'\nInteraction: gap widens from {gap_bg:.1f} pp (background) to {gap_hi:.1f} pp (high CO2)')
print(f'Widening = {gap_hi - gap_bg:.1f} pp  → SF×occupancy interaction confirmed')

t77_long.to_csv(DATA_DIR / '3_table_7_7.csv', index=False)

## 9 · Other Events — Text Results  *(§7.3.2, §7.5)*

Results for events not in numbered tables but reported in Chapter 7 text.  
Includes: E3 (CO₂ rising), E4 (time-of-day), E5 (office hours), E6 (season detail),  
E7 (PM2.5), E8 (PM2.5 tier), E9 (pressure), E10 (pressure drop), E11 (humidity),  
E12 (temperature), E17 (RSSI), E18 (ESP).

In [9]:
def event_comparison(col, g1_val, g0_val=None, label=''):
    """Compare PDR between two groups. g0=None means 'all other values'."""
    g1 = df_link[df_link[col] == g1_val]
    g0 = df_link[df_link[col] != g1_val] if g0_val is None \
         else df_link[df_link[col] == g0_val]
    p1, p0 = pdr(g1), pdr(g0)
    diff   = p1 - p0
    _, p_val, r = mannwhitney(g1['total_tx'].values, g0['total_tx'].values)
    p_str  = '< 0.001' if p_val < 0.001 else f'{p_val:.4f}'
    sig    = '✓' if p_val < 0.05 else '✗'
    print(f'  {label}')
    print(f'    {str(g1_val):<15}: {p1:.2f}%  (n={len(g1):,})')
    print(f'    {str(g0_val or "other"):<15}: {p0:.2f}%  (n={len(g0):,})')
    print(f'    Δ = {diff:+.2f} pp  |  p = {p_str}  |  r = {r:.3f}  {sig}')
    print()
    return {'label': label, 'pdr_g1': round(p1,2), 'pdr_g0': round(p0,2),
            'diff_pp': round(diff,2), 'p_value': p_str, 'r': round(r,3),
            'significant': p_val < 0.05}

other_results = []

print('=' * 65)
print('TEMPORAL EVENTS')
print('=' * 65)

# E5: office hours
print('\nE5: Office Hours vs Off-Peak')
other_results.append(event_comparison('e5_office_hours', 1, 0, 'Office hours (1) vs off-peak (0)'))

# E3: CO2 rising
print('E3: CO2 Rising vs Stable')
other_results.append(event_comparison('e3_co2_rising', 1, 0, 'CO2 rising >20ppm (1) vs stable (0)'))

# E4: time of day
print('E4: PDR by Time of Day')
for band in ['night', 'morning', 'peak', 'evening']:
    g = df_link[df_link['e4_time_of_day'] == band]
    print(f'    {band:<10}: {pdr(g):.2f}%  (burst {burst_rate(g):.1f}%)  n={len(g):,}')

print('\n' + '=' * 65)
print('AIR QUALITY EVENTS')
print('=' * 65)

# E7: PM2.5 spike
print('\nE7: PM2.5 Spike')
g_spike = df_link[df_link['e7_pm25_spike']==1]
g_norm  = df_link[df_link['e7_pm25_spike']==0]
_, p_val, r = mannwhitney(g_spike['total_tx'].values, g_norm['total_tx'].values)
print(f'    Spike     : PDR={pdr(g_spike):.2f}%  burst={burst_rate(g_spike):.1f}%  n={len(g_spike):,}')
print(f'    No spike  : PDR={pdr(g_norm):.2f}%   burst={burst_rate(g_norm):.1f}%  n={len(g_norm):,}')
print(f'    Δ = {pdr(g_spike)-pdr(g_norm):+.2f} pp  |  p = {"< 0.001" if p_val<0.001 else f"{p_val:.4f}"}  |  r = {r:.3f}')

# E8: PM2.5 tier
print('\nE8: PDR by PM2.5 Tier')
for tier in ['clean', 'moderate', 'elevated']:
    g = df_link[df_link['e8_pm25_tier']==tier]
    if len(g)>0: print(f'    {tier:<10}: {pdr(g):.2f}%  n={len(g):,}')

print('\n' + '=' * 65)
print('ATMOSPHERIC EVENTS')
print('=' * 65)

# E9: pressure tier
print('\nE9: PDR by Pressure Tier')
for tier in ['low', 'medium_low', 'medium_high', 'high']:
    g = df_link[df_link['e9_pressure_tier']==tier]
    if len(g)>0: print(f'    {tier:<15}: {pdr(g):.2f}%  n={len(g):,}')

# E10: pressure drop
print('\nE10: Pressure Drop Event')
other_results.append(event_comparison('e10_pressure_drop', 1, 0, 'Pressure drop (1) vs stable (0)'))

# E11: humidity tier
print('E11: PDR by Humidity Tier')
for tier in ['dry', 'normal', 'humid', 'very_humid']:
    g = df_link[df_link['e11_humidity_tier']==tier]
    if len(g)>0: print(f'    {tier:<12}: {pdr(g):.2f}%  n={len(g):,}')
g_dry = df_link[df_link['e11_humidity_tier']=='dry']
g_hum = df_link[df_link['e11_humidity_tier']=='very_humid']
if len(g_hum)>0:
    _, p_val, r = mannwhitney(g_dry['total_tx'].values, g_hum['total_tx'].values)
    print(f'    Dry vs Very Humid: Δ={pdr(g_dry)-pdr(g_hum):+.2f} pp  p={"< 0.001" if p_val<0.001 else f"{p_val:.4f}"}  r={r:.3f}')

# E12: temperature tier
print('\nE12: PDR by Temperature Tier')
for tier in ['cold', 'cool', 'warm', 'hot']:
    g = df_link[df_link['e12_temp_tier']==tier]
    if len(g)>0: print(f'    {tier:<8}: {pdr(g):.2f}%  n={len(g):,}')

print('\n' + '=' * 65)
print('SIGNAL CONTEXT EVENTS')
print('=' * 65)

# E17: RSSI tier
print('\nE17: PDR by RSSI Tier')
for tier in ['weak', 'moderate', 'strong']:
    g = df_link[df_link['e17_rssi_tier']==tier]
    if len(g)>0: print(f'    {tier:<10}: {pdr(g):.2f}%  n={len(g):,}')
g_weak   = df_link[df_link['e17_rssi_tier']=='weak']
g_strong = df_link[df_link['e17_rssi_tier']=='strong']
_, p_val, r = mannwhitney(g_weak['total_tx'].values, g_strong['total_tx'].values)
print(f'    Weak vs Strong: Δ={pdr(g_weak)-pdr(g_strong):+.2f} pp  p={"< 0.001" if p_val<0.001 else f"{p_val:.4f}"}  r={r:.3f}')

# E18: ESP tier
print('\nE18: PDR by ESP Tier')
for tier in ['low_esp', 'medium_esp', 'high_esp']:
    g = df_link[df_link['e18_esp_tier']==tier]
    if len(g)>0: print(f'    {tier:<12}: {pdr(g):.2f}%  n={len(g):,}')

TEMPORAL EVENTS

E5: Office Hours vs Off-Peak
  Office hours (1) vs off-peak (0)
    1              : 95.36%  (n=338,049)
    other          : 96.78%  (n=818,358)
    Δ = -1.42 pp  |  p = < 0.001  |  r = -0.005  ✓

E3: CO2 Rising vs Stable
  CO2 rising >20ppm (1) vs stable (0)
    1              : 33.48%  (n=1,087)
    other          : 96.53%  (n=1,155,320)
    Δ = -63.05 pp  |  p = < 0.001  |  r = -0.323  ✓

E4: PDR by Time of Day
    night     : 96.83%  (burst 0.3%)  n=338,998
    morning   : 96.72%  (burst 0.3%)  n=145,160
    peak      : 95.68%  (burst 0.3%)  n=381,259
    evening   : 96.53%  (burst 0.3%)  n=290,990

AIR QUALITY EVENTS

E7: PM2.5 Spike
    Spike     : PDR=94.28%  burst=0.4%  n=115,186
    No spike  : PDR=96.59%   burst=0.3%  n=1,041,221
    Δ = -2.31 pp  |  p = < 0.001  |  r = -0.015

E8: PDR by PM2.5 Tier
    clean     : 96.65%  n=799,081
    moderate  : 95.99%  n=343,389
    elevated  : 89.36%  n=13,937

ATMOSPHERIC EVENTS

E9: PDR by Pressure Tier
    low       

## 10 · Table 7.8 — Logistic Regression Odds Ratios  *(§7.7 — RQ3)*

Two models fitted:
1. **Loss risk** — outcome: `mac_to_radio_loss > 0` (any loss in interval)
2. **Burst risk** — outcome: `mac_to_radio_loss >= 3` (burst loss, B=3)

Predictors: 18 event indicator columns + device-ID fixed effects (ED0 as reference).  
Odds ratios computed as OR = exp(β). 95% CI via bootstrap (1000 samples, stratified by device).

**Interpretation:**
- OR > 1 → condition increases loss/burst probability
- OR < 1 → condition decreases loss/burst probability

In [10]:
# ── outcome variables
df_link = df_link.copy()
df_link['y_loss']  = (df_link['mac_to_radio_loss'] > 0).astype(int)   # any loss
df_link['y_burst'] = (df_link['mac_to_radio_loss'] >= 3).astype(int)  # burst B=3

print(f'Loss  rate (y_loss=1) : {df_link["y_loss"].mean()*100:.2f}%')
print(f'Burst rate (y_burst=1): {df_link["y_burst"].mean()*100:.2f}%')

# ── feature matrix — categorical events as dummies, reference categories match thesis
cat_cols = {
    'e1_co2_tier':      'background',
    'e4_time_of_day':   'night',
    'e6_season':        'winter',
    'e8_pm25_tier':     'clean',
    'e9_pressure_tier': 'low',
    'e11_humidity_tier':'dry',
    'e12_temp_tier':    'cold',
    'e14_sf_tier':      'low_sf',
    'e15_toa_class':    'short_toa',
    'e16_loss_type':    'no_loss',
    'e17_rssi_tier':    'strong',
    'e18_esp_tier':     'high_esp',
    'device_id':        'ED0',
}
bin_cols = ['e2_is_weekday', 'e3_co2_rising', 'e5_office_hours',
            'e7_pm25_spike', 'e10_pressure_drop', 'e13_sf']

# convert categorical to string for pd.get_dummies
df_feat = df_link.copy()
for col in cat_cols:
    df_feat[col] = df_feat[col].astype(str)

# build dummies — drop reference category for each
X_cat = pd.DataFrame()
for col, ref in cat_cols.items():
    dummies = pd.get_dummies(df_feat[col], prefix=col)
    ref_col = f'{col}_{ref}'
    if ref_col in dummies.columns:
        dummies = dummies.drop(columns=[ref_col])
    X_cat = pd.concat([X_cat, dummies], axis=1)

X_bin = df_feat[bin_cols].fillna(0).astype(float)
X     = pd.concat([X_cat, X_bin], axis=1).fillna(0).astype(float)
feat_names = X.columns.tolist()

print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')

Loss  rate (y_loss=1) : 1.85%
Burst rate (y_burst=1): 0.28%
Feature matrix: 1,156,407 rows × 37 features


In [11]:
# ── fit logistic regression with bootstrap 95% CI
def fit_lr_bootstrap(X: pd.DataFrame, y: pd.Series, n_boot: int = 1000):
    """Fit LR, compute OR and 95% bootstrap CI (stratified by device)."""
    model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
    model.fit(X, y)
    auc   = roc_auc_score(y, model.predict_proba(X)[:, 1])
    coefs = model.coef_[0]

    boot_coefs = []
    for _ in range(n_boot):
        X_b, y_b = resample(X, y, stratify=df_link['device_id'].values, random_state=None)
        m = LogisticRegression(max_iter=500, solver='lbfgs', C=1.0)
        m.fit(X_b, y_b)
        boot_coefs.append(m.coef_[0])

    boot_coefs = np.array(boot_coefs)
    ci_lo = np.exp(np.percentile(boot_coefs, 2.5,  axis=0))
    ci_hi = np.exp(np.percentile(boot_coefs, 97.5, axis=0))
    ors   = np.exp(coefs)

    result = pd.DataFrame({
        'Feature': feat_names,
        'OR':      ors.round(3),
        'CI_lo':   ci_lo.round(3),
        'CI_hi':   ci_hi.round(3),
        'coef':    coefs.round(4),
    }).sort_values('OR', ascending=False).reset_index(drop=True)
    return result, auc

y_loss  = df_link['y_loss'].values
y_burst = df_link['y_burst'].values

print('Fitting loss risk model (n_boot=1000) ...')
loss_res, auc_loss = fit_lr_bootstrap(X, y_loss)
print(f'  AUC (loss risk) = {auc_loss:.3f}')

print('Fitting burst risk model (n_boot=1000) ...')
burst_res, auc_burst = fit_lr_bootstrap(X, y_burst)
print(f'  AUC (burst risk) = {auc_burst:.3f}')

Fitting loss risk model (n_boot=1000) ...


KeyboardInterrupt: 

In [12]:
# ── Table 7.8 — exclude device fixed effects from display
# device effects are controls, not the focus of the thesis
mask = ~loss_res['Feature'].str.startswith('device_id_')

print('TABLE 7.8: Logistic Regression Odds Ratios')
print(f"{'Feature':<40} {'OR_loss':>9} {'CI_loss':>16} {'OR_burst':>10} {'CI_burst':>16}")
print('-' * 95)

# merge loss and burst results
t78 = loss_res[mask][['Feature','OR','CI_lo','CI_hi']].copy()
t78.columns = ['Feature','OR_loss','loss_CI_lo','loss_CI_hi']
t78_b = burst_res[mask][['Feature','OR','CI_lo','CI_hi']].copy()
t78_b.columns = ['Feature','OR_burst','burst_CI_lo','burst_CI_hi']
t78_full = t78.merge(t78_b, on='Feature').sort_values('OR_loss', ascending=False)

for _, row in t78_full.iterrows():
    ci_l = f'[{row["loss_CI_lo"]:.3f}, {row["loss_CI_hi"]:.3f}]'
    ci_b = f'[{row["burst_CI_lo"]:.3f}, {row["burst_CI_hi"]:.3f}]'
    print(f"{row['Feature']:<40} {row['OR_loss']:>9.3f} {ci_l:>16} {row['OR_burst']:>10.3f} {ci_b:>16}")

print(f'\nAUC (loss risk): {auc_loss:.3f}  |  AUC (burst risk): {auc_burst:.3f}')

# top 5 strongest predictors each model
print('\nTop 5 loss-risk predictors:')
for _, r in t78_full.head(5).iterrows():
    print(f"  {r['Feature']:<40}: OR = {r['OR_loss']:.3f}  [{r['loss_CI_lo']:.3f}, {r['loss_CI_hi']:.3f}]")
print('\nTop 5 burst-risk predictors:')
for _, r in t78_full.sort_values('OR_burst', ascending=False).head(5).iterrows():
    print(f"  {r['Feature']:<40}: OR = {r['OR_burst']:.3f}  [{r['burst_CI_lo']:.3f}, {r['burst_CI_hi']:.3f}]")

t78_full.to_csv(DATA_DIR / '3_table_7_8.csv', index=False)

NameError: name 'loss_res' is not defined

## 11 · Research Question Answers Summary

Consolidates all results into direct answers to RQ1, RQ2, RQ3.

In [13]:
print('=' * 65)
print('RESEARCH QUESTION ANSWERS')
print('=' * 65)

print('\nRQ1: Campaign-wide reliability')
print(f'  PDR_link       : {camp_pdr:.2f}%')
print(f'  PDR range      : {pdr_range:.2f} pp across 6 devices — collision evidence')
print(f'  Large burst rate: {pcts["large_burst"]:.1f}% of intervals')

print('\nRQ2: Event-conditioned reliability (key findings)')
bg_co2 = df_link[df_link['e1_co2_tier']=='background']
hi_co2 = df_link[df_link['e1_co2_tier']=='high']
print(f'  CO2: background→high PDR drop: {pdr(bg_co2):.1f}% → {pdr(hi_co2):.1f}%  ({pdr(hi_co2)-pdr(bg_co2):+.1f} pp)')
sf7_g = df_link[df_link['e13_sf']==7]
sf10_g= df_link[df_link['e13_sf']==10]
print(f'  SF7→SF10 PDR drop: {pdr(sf7_g):.1f}% → {pdr(sf10_g):.1f}%  ({pdr(sf10_g)-pdr(sf7_g):+.1f} pp)')
print(f'  SF×CO2 gap widens: {gap_bg:.1f} pp (background) → {gap_hi:.1f} pp (high CO2)')

print('\nRQ3: Strongest predictors (loss risk OR)')
for _, r in t78_full.head(3).iterrows():
    print(f'  {r["Feature"]:<40}: OR = {r["OR_loss"]:.3f}')

RESEARCH QUESTION ANSWERS

RQ1: Campaign-wide reliability
  PDR_link       : 96.36%
  PDR range      : 0.81 pp across 6 devices — collision evidence
  Large burst rate: 0.2% of intervals

RQ2: Event-conditioned reliability (key findings)
  CO2: background→high PDR drop: 96.8% → 94.6%  (-2.3 pp)
  SF7→SF10 PDR drop: 98.1% → 91.4%  (-6.7 pp)


NameError: name 'gap_bg' is not defined

## 12 · Save

In [ ]:
saved = [
    '3_table_7_1.csv', '3_table_7_2.csv', '3_table_7_3.csv',
    '3_table_7_4.csv', '3_table_7_5.csv', '3_table_7_6.csv',
    '3_table_7_7.csv', '3_table_7_8.csv',
]
for f in saved:
    print(f'Saved: {DATA_DIR / f}')